# 🎬 SadTalker — Lip Sync pour Réseaux Sociaux

Génère une vidéo animée avec synchronisation des lèvres à partir d'une **image PNG** et d'un **fichier audio MP3/WAV**.
Export automatique au format **vertical 9:16** (TikTok, Instagram Reels, YouTube Shorts).

---

| Étape | Description |
|---|---|
| 🔧 **1** | Vérification GPU + installation des dépendances |
| 📦 **2** | Clonage de SadTalker + téléchargement des modèles |
| 🖼️ **3** | Upload de l'image source (PNG) |
| 🎵 **4** | Upload de l'audio (MP3 / WAV) |
| 🚀 **5** | Génération de la vidéo |
| 📱 **6** | Export 9:16 + téléchargement |

> **Prérequis :** Activer le GPU T4 dans *Exécution → Modifier le type d'exécution → GPU*

---
## 🔧 Étape 1 — Vérification du GPU et installation

In [ ]:
# ─── Vérification du GPU ───────────────────────────────────────────────────────
# SadTalker nécessite obligatoirement un GPU pour une génération rapide.
# Si 'No GPU' apparaît → Exécution > Modifier le type d'exécution > GPU T4

import subprocess, sys

def run(cmd):
    """Exécute une commande shell et affiche la sortie."""
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0 and result.stderr:
        print("[ERREUR]", result.stderr[-2000:])
    return result.returncode == 0

print("=" * 60)
print("  VÉRIFICATION DU MATÉRIEL")
print("=" * 60)

try:
    import torch
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        vram = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"✅ GPU détecté : {name}")
        print(f"   VRAM        : {vram:.1f} GB")
        print(f"   CUDA        : {torch.version.cuda}")
    else:
        print("❌ Aucun GPU détecté ! Activez le GPU avant de continuer.")
        sys.exit()
except ImportError:
    print("⚠️  PyTorch non encore importé — normal au premier lancement.")

run("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader")
print("=" * 60)

In [ ]:
# ─── Installation des dépendances système ─────────────────────────────────────
# ffmpeg  : conversion et formatage vidéo (obligatoire pour l'export 9:16)
# git-lfs : téléchargement des gros fichiers de modèles depuis Hugging Face

print("📦 Installation des outils système...")
run("apt-get update -qq && apt-get install -y -qq ffmpeg git-lfs libgl1")
run("git lfs install")
print("✅ Outils système installés.")

---
## 📦 Étape 2 — Clonage de SadTalker et téléchargement des modèles

In [ ]:
# ─── Clonage du dépôt SadTalker ───────────────────────────────────────────────
# On se place dans /content pour que tout soit dans le répertoire de travail Colab.
# Le flag --depth 1 télécharge uniquement le dernier commit (plus rapide).

import os

SADTALKER_DIR = "/content/SadTalker"
os.chdir("/content")

if not os.path.isdir(SADTALKER_DIR):
    print("🔽 Clonage de SadTalker...")
    ok = run("git clone --depth 1 https://github.com/OpenTalker/SadTalker.git")
    if ok:
        print("✅ SadTalker cloné avec succès.")
    else:
        print("❌ Erreur lors du clonage — vérifiez votre connexion internet.")
else:
    print("✅ SadTalker déjà présent, on passe.")

os.chdir(SADTALKER_DIR)

In [ ]:
# ─── Correctifs de compatibilité ─────────────────────────────────────────────
# 
# SadTalker a été développé en 2022-2023 et ne s'exécute pas "as-is" sur Colab 2024+
# en raison de breaking changes dans les dépendances. Ce bloc applique trois patches
# qui modifient le code source de SadTalker ET ses dépendances.
#
# PATCH 1 — NumPy 2.x (Sept 2024)
#   Symptôme : AttributeError: module 'numpy' has no attribute 'VisibleDeprecationWarning'
#   Cause : NumPy 2.0 supprima les types deprecated np.bool, np.int, np.float, etc.
#           pour forcer migration vers les builtins Python (bool, int, float)
#   Fix : Regex-replace de np.bool→bool, np.int→int, np.float→float, etc.
#   Ref : https://numpy.org/devdocs/release/2.0.0-notes.html#numpy-2-0-0
#
# PATCH 2 — torchvision 0.16+ (Juin 2024)
#   Symptôme : ModuleNotFoundError: No module named 'torchvision.transforms.functional_tensor'
#   Cause : torchvision 0.16 supprima le module interne functional_tensor
#           qui était un détail d'implémentation, pas une API publique
#   Fix : (1) Sed-replace functional_tensor→functional dans basicsr/degradations.py
#         (2) Shim sitecustomize.py qui injecte un module fake dans sys.modules
#   Ref : https://github.com/pytorch/vision/releases/tag/v0.16.0
#
# PATCH 3 — PyTorch 2.0+ (Mars 2023)
#   Symptôme : EOFError lors du torch.load() des fichiers .pth legacy
#   Cause : PyTorch 2.0 défaut à weights_only=True (mode sécurisé restrictif)
#           qui utilise unpickler custom incompatible avec anciens .pickle files
#           SadTalker utilise des modèles entraînés avec PyTorch 1.x
#   Fix : AST-based patching pour ajouter weights_only=False à chaque torch.load()
#   Ref : https://pytorch.org/docs/2.0/generated/torch.load.html
#
# Alternatives rejetées :
#   • Downgrader PyTorch → cassera Colab et autres tenseurs
#   • Réentraîner modèles SadTalker → coûteux, unnecessary
#   • Refactoriser SadTalker → diverge de upstream, maintenance cauchemar
#   → Patching in-place est le meilleur compromis : minimal et isolé

import os, re, glob, subprocess, sys, sysconfig, ast

SADTALKER_SRC = "/content/SadTalker"

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).returncode == 0

# ── Patch 1 : NumPy 2.x ───────────────────────────────────────────────────────
NUMPY_PATTERNS = [
    (r"np\.VisibleDeprecationWarning", "DeprecationWarning"),
    (r"\bnp\.bool\b(?!\w)",    "bool"),
    (r"\bnp\.int\b(?!\w)",     "int"),
    (r"\bnp\.float\b(?!\w)",   "float"),
    (r"\bnp\.complex\b(?!\w)", "complex"),
    (r"\bnp\.object\b(?!\w)",  "object"),
    (r"\bnp\.str\b(?!\w)",     "str"),
]
numpy_patched = []
for filepath in glob.glob(f"{SADTALKER_SRC}/**/*.py", recursive=True):
    try:
        src = open(filepath, "r", encoding="utf-8", errors="ignore").read()
        mod = src
        for pat, rep in NUMPY_PATTERNS:
            mod = re.sub(pat, rep, mod)
        if mod != src:
            open(filepath, "w", encoding="utf-8").write(mod)
            numpy_patched.append(os.path.relpath(filepath, SADTALKER_SRC))
    except Exception as e:
        print(f"  ⚠️  {filepath} : {e}")
print(f"✅ NumPy 2.x — {len(numpy_patched)} fichier(s) patché(s)")

# ── Patch 2 : torchvision functional_tensor ───────────────────────────────────
OLD_TV = "from torchvision.transforms.functional_tensor import rgb_to_grayscale"
NEW_TV = "from torchvision.transforms.functional import rgb_to_grayscale"

degradations = None
try:
    import basicsr
    degradations = os.path.join(os.path.dirname(basicsr.__file__), "data", "degradations.py")
except ImportError:
    for p in sys.path:
        c = os.path.join(p, "basicsr", "data", "degradations.py")
        if os.path.isfile(c):
            degradations = c; break
if not degradations:
    r = subprocess.run("find /usr /opt /root -name 'degradations.py' -path '*/basicsr/*' 2>/dev/null | head -1",
                       shell=True, capture_output=True, text=True)
    if r.stdout.strip():
        degradations = r.stdout.strip()

if degradations and os.path.isfile(degradations):
    content = open(degradations).read()
    if OLD_TV in content:
        sh(f"sed -i 's|{OLD_TV}|{NEW_TV}|g' \"{degradations}\"")
        print("✅ torchvision compat — degradations.py patché")
    else:
        print("✅ torchvision compat — déjà corrigé")
else:
    print("⚠️  basicsr/degradations.py introuvable — shim sitecustomize actif")

SITECUSTOMIZE = os.path.join(sysconfig.get_path("stdlib"), "sitecustomize.py")
SHIM_MARKER = "# sadtalker-torchvision-shim"
SHIM_CODE = f"""
{SHIM_MARKER}
try:
    import sys, types
    import torchvision.transforms.functional as _tvf
    _m = types.ModuleType("torchvision.transforms.functional_tensor")
    _m.rgb_to_grayscale = _tvf.rgb_to_grayscale
    sys.modules.setdefault("torchvision.transforms.functional_tensor", _m)
except Exception:
    pass
"""
existing = open(SITECUSTOMIZE).read() if os.path.isfile(SITECUSTOMIZE) else ""
if SHIM_MARKER not in existing:
    open(SITECUSTOMIZE, "a").write(SHIM_CODE)
    print("✅ sitecustomize.py — shim ajouté")
else:
    print("✅ sitecustomize.py — shim actif")

# ── Patch 3 : PyTorch 2.x — weights_only=False ────────────────────────────────
# Utilise AST (Abstract Syntax Tree) pour une analyse robuste et correcte
# des appels torch.load(), même imbriqués et complexes.
# Regex précédent s'arrêtait à la 1ère ')' et cassait torch.load(..., map_location=torch.device(...))

class TorchLoadPatcher(ast.NodeTransformer):
    def visit_Call(self, node):
        self.generic_visit(node)
        if isinstance(node.func, ast.Attribute) and isinstance(node.func.value, ast.Name):
            if node.func.value.id == "torch" and node.func.attr == "load":
                kw_names = [kw.arg for kw in node.keywords if kw.arg]
                if "weights_only" not in kw_names:
                    node.keywords.append(
                        ast.keyword(arg="weights_only", value=ast.Constant(value=False))
                    )
        return node

torch_patched = []
for filepath in glob.glob(f"{SADTALKER_SRC}/**/*.py", recursive=True):
    try:
        src = open(filepath, "r", encoding="utf-8", errors="ignore").read()
        if "torch.load(" not in src:
            continue
        tree = ast.parse(src)
        patcher = TorchLoadPatcher()
        new_tree = patcher.visit(tree)
        ast.fix_missing_locations(new_tree)
        # Utiliser ast.unparse (Python 3.9+) pour convertir AST → code source
        try:
            new_src = ast.unparse(new_tree)
        except AttributeError:
            # Python 3.8 n'a pas ast.unparse → utiliser astor (installé via pip)
            import astor
            new_src = astor.to_source(new_tree)
        if new_src != src:
            open(filepath, "w", encoding="utf-8").write(new_src)
            torch_patched.append(os.path.relpath(filepath, SADTALKER_SRC))
    except SyntaxError:
        pass
    except Exception as e:
        print(f"  ⚠️  {filepath} : {e}")
print(f"✅ PyTorch 2.x weights_only — {len(torch_patched)} fichier(s) patché(s)")

In [ ]:
# ─── Installation des dépendances Python ──────────────────────────────────────
# Colab fournit déjà PyTorch + CUDA — on ne le réinstalle pas.
# Chaque paquet est installé séparément pour isoler les erreurs.

import subprocess, sys, time

def run(cmd, label="", show_output=False):
    t = time.time()
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    ok = result.returncode == 0
    elapsed = time.time() - t
    if label:
        print(f"  {'✅' if ok else '⚠️ '} {label} ({elapsed:.0f}s)")
    if show_output and result.stdout:
        print(result.stdout[-500:])
    if not ok and result.stderr:
        last = [l for l in result.stderr.splitlines() if l.strip()]
        if last:
            print(f"     → {last[-1]}")
    return ok

print("🐍 Installation des dépendances SadTalker...\n")

# Paquets Python — dlib retiré (non utilisé par SadTalker, compile trop longtemps)
PACKAGES = [
    ("scipy",                 "scipy"),
    ("Pillow",                "Pillow"),
    ("imageio",               "imageio"),
    ("imageio-ffmpeg",        "imageio-ffmpeg"),
    ("librosa",               "librosa"),
    ("resampy",               "resampy"),
    ("pydub",                 "pydub"),
    ("yacs",                  "yacs"),
    ("tqdm",                  "tqdm"),
    ("einops",                "einops"),
    ("av",                    "av (PyAV)"),
    ("kornia",                "kornia"),
    ("basicsr",               "basicsr"),
    ("facexlib",              "facexlib"),
    ("gfpgan",                "gfpgan"),
    ("face_alignment",        "face_alignment"),
    ("safetensors",           "safetensors"),
    ("huggingface_hub",       "huggingface_hub"),
    ("batch_face",            "batch_face"),
]

failed = []
for pkg, label in PACKAGES:
    ok = run(f"pip install -q '{pkg}'", label)
    if not ok:
        failed.append(pkg)

print()
if failed:
    print(f"⚠️  {len(failed)} paquet(s) en échec (souvent non bloquants) :")
    for p in failed:
        print(f"   • {p}")
else:
    print("✅ Toutes les dépendances installées.")

import torch, numpy as np
print(f"\n📋 Environnement :")
print(f"   PyTorch : {torch.__version__} | CUDA : {torch.version.cuda}")
print(f"   NumPy   : {np.__version__}")
assert torch.cuda.is_available(), "❌ GPU non disponible — activez le GPU T4."

In [ ]:
# ─── Téléchargement des modèles pré-entraînés ─────────────────────────────────
import os, shutil, hashlib, time, subprocess, glob
from huggingface_hub import snapshot_download, hf_hub_download

def retry_with_backoff(fn, max_retries=Config.DOWNLOAD_RETRIES, label=""):
    """Réessaye une fonction avec backoff exponentiel (2s, 4s, 8s, 16s)."""
    for attempt in range(max_retries):
        try:
            return fn()
        except Exception as e:
            if attempt == max_retries - 1:
                raise
            wait = 2 ** attempt
            print(f"  ⚠️  Tentative {attempt + 1} échouée ({label}): {str(e)[:100]}")
            print(f"     Réessai dans {wait}s...")
            time.sleep(wait)

def validate_checksum(filepath, expected_sha=None, min_size=10000):
    """Valide qu'un fichier existe, a une taille minimale, et optionnellement son SHA256."""
    if not os.path.isfile(filepath):
        return False, "fichier inexistant"
    size = os.path.getsize(filepath)
    if size < min_size:
        return False, f"fichier trop petit ({size} o < {min_size})"
    if expected_sha:
        sha = hashlib.sha256(open(filepath, "rb").read()).hexdigest()
        if sha != expected_sha:
            return False, f"checksum incorrect"
    return True, size

# ── Étape 1 : snapshot_download du repo vinthony/SadTalker ────────────────────
if not os.path.isdir(Config.HF_CACHE) or not os.listdir(Config.HF_CACHE):
    print("🤖 Téléchargement du repo SadTalker (10-20 min)...\n")
    try:
        snapshot_download(
            repo_id="vinthony/SadTalker",
            local_dir=Config.HF_CACHE,
            ignore_patterns=["*.md", "*.txt", "*.py"],
        )
        print(f"✅ Repo téléchargé dans {Config.HF_CACHE}")
    except Exception as e:
        print(f"❌ snapshot_download : {e}")
else:
    print(f"✅ Repo déjà en cache : {Config.HF_CACHE}")

# ── Étape 2 : copie vers les répertoires SadTalker ────────────────────────────
if os.path.isdir(Config.HF_CACHE):
    print("\n📁 Organisation des fichiers...")
    copied = 0
    for root, dirs, files in os.walk(Config.HF_CACHE):
        for fname in files:
            if fname.endswith(".metadata") or fname in [".gitattributes", ".gitignore",
                                                         "CACHEDIR.TAG", ".DS_Store"]:
                continue
            src = os.path.join(root, fname)
            rel = os.path.relpath(root, Config.HF_CACHE)
            dst = os.path.join(Config.BFM_DIR if "BFM_Fitting" in rel else Config.CHECKPOINT_DIR, fname)
            if not os.path.isfile(dst):
                shutil.copy2(src, dst)
                print(f"  ✓ {fname}")
                copied += 1
    print(f"  → {copied} fichier(s) copié(s)")

# ── Étape 3 : BFM_model_front.mat ────────────────────────────────────────────
# Ce fichier n'existe PAS dans le HF repo vinthony/SadTalker.
# Source officielle : SadTalker v0.0.2 GitHub releases (BFM_Fitting.zip)
# Contient : BFM_model_front.mat et d'autres fichiers du Basel Face Model.
BFM_FRONT = f"{Config.BFM_DIR}/BFM_model_front.mat"

ok_bfm, _ = validate_checksum(BFM_FRONT, min_size=100_000)
if not ok_bfm:
    print("\n📐 Téléchargement de BFM_Fitting.zip (GitHub releases)...")

    # 1. Chercher dans le cache HF existant (glob récursif)
    found_in_cache = glob.glob(f"{Config.HF_CACHE}/**/BFM_model_front.mat", recursive=True)
    if found_in_cache:
        shutil.copy2(found_in_cache[0], BFM_FRONT)
        print(f"  ✅ Trouvé dans le cache HF")
    else:
        # 2. Télécharger BFM_Fitting.zip depuis GitHub releases v0.0.2
        BFM_ZIP_URL = "https://github.com/Winfredy/SadTalker/releases/download/v0.0.2/BFM_Fitting.zip"
        BFM_ZIP_PATH = "/tmp/BFM_Fitting.zip"
        
        dl_result = subprocess.run(
            f'wget -q --timeout=120 --tries=3 "{BFM_ZIP_URL}" -O "{BFM_ZIP_PATH}"',
            shell=True, capture_output=True, text=True, timeout=180
        )
        
        if dl_result.returncode == 0 and os.path.isfile(BFM_ZIP_PATH) and os.path.getsize(BFM_ZIP_PATH) > 100_000:
            print(f"  📦 BFM_Fitting.zip téléchargé ({os.path.getsize(BFM_ZIP_PATH)//1000} KB)")
            extract_dir = "/tmp/bfm_extract"
            os.makedirs(extract_dir, exist_ok=True)
            subprocess.run(f'unzip -q -o "{BFM_ZIP_PATH}" -d "{extract_dir}"',
                           shell=True, capture_output=True)
            # Copier tous les fichiers extraits vers BFM_DIR
            for f in glob.glob(f"{extract_dir}/**/*", recursive=True):
                if os.path.isfile(f):
                    dst = os.path.join(Config.BFM_DIR, os.path.basename(f))
                    if not os.path.isfile(dst):
                        shutil.copy2(f, dst)
                        print(f"  ✓ {os.path.basename(f)}")
        else:
            print(f"  ⚠️  GitHub releases inaccessible — tentative source alternative...")
            # 3. Fallback : essayer le repo GitHub OpenTalker/SadTalker-V2 ou autre source HF connue
            ALT_SOURCES = [
                # Certains forks/mirrors hébergent le fichier directement
                ("hf", "vinthony/SadTalker", "BFM_model_front.mat"),
                ("hf", "vinthony/SadTalker", "checkpoints/BFM_Fitting/BFM_model_front.mat"),
            ]
            for kind, repo_id, filename in ALT_SOURCES:
                try:
                    if kind == "hf":
                        hf_hub_download(repo_id=repo_id, filename=filename,
                                        local_dir=Config.BFM_DIR,
                                        local_dir_use_symlinks=False)
                        # Déplacer si créé dans un sous-dossier
                        for match in glob.glob(f"{Config.BFM_DIR}/**/BFM_model_front.mat", recursive=True):
                            if match != BFM_FRONT:
                                shutil.move(match, BFM_FRONT)
                        if os.path.isfile(BFM_FRONT) and os.path.getsize(BFM_FRONT) > 100_000:
                            print(f"  ✅ Téléchargé depuis {repo_id}/{filename}")
                            break
                except Exception as e:
                    print(f"  ⚠️  {repo_id}/{filename} : {str(e)[:80]}")

    ok_bfm, msg = validate_checksum(BFM_FRONT, min_size=100_000)
    sz = os.path.getsize(BFM_FRONT) if os.path.isfile(BFM_FRONT) else 0
    print(f"  {'✅' if ok_bfm else '❌'} BFM_model_front.mat ({sz//1_000_000} MB)")
    if not ok_bfm:
        print("  💡 Si ce fichier manque, SadTalker peut quand même fonctionner")
        print("     avec un mode de prétraitement alternatif (PREPROCESS='full').")
else:
    print(f"\n✅ BFM_model_front.mat ({os.path.getsize(BFM_FRONT)//1_000_000} MB)")

# ── Étape 4 : vérification des fichiers critiques ────────────────────────────
REQUIRED = {
    f"{Config.CHECKPOINT_DIR}/epoch_20.pth":                    (20_000_000, "alignment model"),
    f"{Config.CHECKPOINT_DIR}/facevid2vid_00189-model.pth.tar": (200_000_000, "face renderer"),
    f"{Config.CHECKPOINT_DIR}/auido2exp_00300-model.pth":       (5_000_000,  "audio→expression"),
    f"{Config.CHECKPOINT_DIR}/auido2pose_00140-model.pth":      (5_000_000,  "audio→pose"),
    f"{Config.BFM_DIR}/BFM_model_front.mat":                    (100_000,    "3D face model"),
    f"{Config.BFM_DIR}/01_MorphableModel.mat":                  (100_000,    "morphable model"),
    f"{Config.BFM_DIR}/Exp_Pca.bin":                            (1_000_000,  "expression PCA"),
}
print("\n🔍 Vérification des fichiers critiques...")
missing = []
for path, (min_sz, label) in REQUIRED.items():
    ok, msg = validate_checksum(path, min_size=min_sz)
    sz = os.path.getsize(path) if os.path.isfile(path) else 0
    print(f"  {'✅' if ok else '❌'} {os.path.basename(path)} — {label} ({sz//1_000_000} MB)")
    if not ok:
        missing.append(os.path.basename(path))

# ── Étape 5 : modèles GFPGAN (optionnels)
print("\n🎨 Modèles GFPGAN (optionnels)...")
try:
    from basicsr.utils.download_util import load_file_from_url
    GFPGAN_URLS = {
        "GFPGANv1.4.pth":
            "https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth",
        "alignment_WFLW_4HG.pth":
            "https://github.com/xinntao/facexlib/releases/download/v0.1.0/alignment_WFLW_4HG.pth",
        "detection_Resnet50_Final.pth":
            "https://github.com/xinntao/facexlib/releases/download/v0.1.0/detection_Resnet50_Final.pth",
        "parsing_parsenet.pth":
            "https://github.com/xinntao/facexlib/releases/download/v0.2.2/parsing_parsenet.pth",
    }
    for fname, url in GFPGAN_URLS.items():
        dest = f"{Config.GFPGAN_DIR}/{fname}"
        ok, msg = validate_checksum(dest, min_size=1_000_000)
        if ok:
            print(f"  ✓ {fname} ({os.path.getsize(dest)//1_000_000} MB)")
            continue
        print(f"  ⬇️  {fname}...")
        try:
            def dl_gfpgan():
                load_file_from_url(url, model_dir=Config.GFPGAN_DIR, progress=True, file_name=fname)
            retry_with_backoff(dl_gfpgan, label=fname)
            ok, msg = validate_checksum(dest, min_size=1_000_000)
            sz = os.path.getsize(dest) if os.path.isfile(dest) else 0
            print(f"  {'✅' if ok else '❌'} {fname} ({sz//1_000_000} MB)")
        except Exception as e:
            print(f"  ⚠️  {fname} : {str(e)[:100]}")
except ImportError:
    print("  ⚠️  basicsr absent → GFPGAN non disponible")

print()
if missing:
    print(f"⚠️  Fichiers manquants : {missing}")
    if "BFM_model_front.mat" in missing:
        print("   → BFM_model_front.mat manquant : essayez Config.PREPROCESS='full' à l'étape 5")
    other_missing = [f for f in missing if f != "BFM_model_front.mat"]
    if other_missing:
        print(f"   → Fichiers critiques manquants (bloquants) : {other_missing}")
else:
    print("✅ Tous les modèles critiques sont présents !")

---
## 🖼️ Étape 3 — Upload de l'image source

In [ ]:
# ─── Upload de l'image PNG ────────────────────────────────────────────────────
# Conseils pour une meilleure qualité :
#   • Image carrée recommandée (ex: 512×512 ou 1024×1024)
#   • Visage bien centré, bien éclairé, de face ou légèrement de profil
#   • Fond uni de préférence pour un meilleur résultat
#   • Formats acceptés : PNG, JPG

import os, shutil
from google.colab import files
from IPython.display import display, Image as IPImage
import ipywidgets as widgets

print("🖼️  Choisissez votre image source (PNG/JPG) :")
print("   → Visage bien visible, centré, fond uni de préférence")
print()

uploaded_img = files.upload()

if not uploaded_img:
    raise FileNotFoundError("❌ Aucun fichier uploadé. Relancez cette cellule.")

img_filename = list(uploaded_img.keys())[0]
img_ext      = os.path.splitext(img_filename)[1].lower()

if img_ext not in [".png", ".jpg", ".jpeg"]:
    raise ValueError(f"❌ Format non supporté : {img_ext}. Utilisez PNG ou JPG.")

SOURCE_IMAGE = f"{Config.INPUT_DIR}/source_image{img_ext}"
shutil.move(img_filename, SOURCE_IMAGE)

print(f"✅ Image chargée : {img_filename}")

from PIL import Image as PILImage
img = PILImage.open(SOURCE_IMAGE)
print(f"   Dimensions : {img.width} × {img.height} px | Mode : {img.mode}")

if img.width < 256 or img.height < 256:
    print("⚠️  Image petite (< 256px) — la qualité peut être réduite.")

display(IPImage(SOURCE_IMAGE, width=300))

---
## 🎵 Étape 4 — Upload de l'audio

In [ ]:
# ─── Upload du fichier audio ──────────────────────────────────────────────────
# Conseils pour un meilleur lip sync :
#   • Voix claire, sans bruit de fond si possible
#   • Durée conseillée : 5 à 30 secondes (au-delà la génération est plus longue)
#   • Formats acceptés : MP3, WAV, M4A
#   • Le fichier sera automatiquement converti en WAV 16kHz mono

import os, shutil, subprocess
from google.colab import files
from IPython.display import Audio, display

print("🎵 Choisissez votre fichier audio (MP3 / WAV / M4A) :")
print("   → Voix claire, durée recommandée : 5-30 secondes")
print()

uploaded_audio = files.upload()

if not uploaded_audio:
    raise FileNotFoundError("❌ Aucun fichier uploadé. Relancez cette cellule.")

audio_filename = list(uploaded_audio.keys())[0]
audio_ext      = os.path.splitext(audio_filename)[1].lower()

if audio_ext not in [".mp3", ".wav", ".m4a", ".ogg", ".flac"]:
    raise ValueError(f"❌ Format non supporté : {audio_ext}. Utilisez MP3, WAV ou M4A.")

raw_audio = f"{Config.INPUT_DIR}/audio_raw{audio_ext}"
shutil.move(audio_filename, raw_audio)

DRIVEN_AUDIO = f"{Config.INPUT_DIR}/driven_audio.wav"
print("🔄 Conversion en WAV 16kHz mono...")
result = subprocess.run(
    f'ffmpeg -y -i "{raw_audio}" -ar 16000 -ac 1 -f wav "{DRIVEN_AUDIO}" -loglevel error',
    shell=True, capture_output=True, text=True
)

if result.returncode != 0:
    raise RuntimeError(f"❌ Conversion audio échouée : {result.stderr}")

dur_raw = subprocess.check_output(
    f'ffprobe -v error -show_entries format=duration -of csv=p=0 "{DRIVEN_AUDIO}"',
    shell=True
).decode().strip()
duration = float(dur_raw) if dur_raw else 0

print(f"✅ Audio prêt : {audio_filename} ({duration:.1f} s)")
if duration > 60:
    print("⚠️  Audio long (> 60s) : la génération peut prendre plusieurs minutes.")

display(Audio(DRIVEN_AUDIO))

---
## ⚙️ Étape 5 — Paramètres de génération

In [ ]:
# ─── Configuration centralisée ────────────────────────────────────────────────
# Tous les paramètres configurables au même endroit pour faciliter la maintenance.

class Config:
    # Chemins
    SADTALKER_DIR = "/content/SadTalker"
    CHECKPOINT_DIR = "/content/SadTalker/checkpoints"
    BFM_DIR = "/content/SadTalker/checkpoints/BFM_Fitting"
    GFPGAN_DIR = "/content/SadTalker/gfpgan/weights"
    OUTPUT_DIR = "/content/outputs"
    INPUT_DIR = "/content/inputs"
    HF_CACHE = "/content/hf_sadtalker"
    
    # Génération
    SIZE = "512"  # "256" rapide (~2 min) | "512" qualité (~5 min)
    PREPROCESS = "crop"  # "crop" optimal lip-sync | "resize" | "full"
    ENHANCE_FACE = False  # Activez si GFPGAN est disponible
    BG_ENHANCER = False  # Nécessite GFPGAN
    
    # Export
    EXPORT_RESOLUTION = "1080x1920"  # "1080x1920" Full HD | "720x1280" HD
    BACKGROUND = "blur"  # "blur" cinématique | "black" | "white"
    
    # Timeouts
    INFERENCE_TIMEOUT = 600  # 10 min pour la génération
    FFMPEG_TIMEOUT = 600    # 10 min pour chaque conversion
    DOWNLOAD_TIMEOUT = 300  # 5 min par téléchargement
    DOWNLOAD_RETRIES = 4    # Nombre de tentatives avec backoff
    
    # Validation
    MIN_MODEL_SIZE = {
        "epoch_20.pth": 20_000_000,
        "facevid2vid_00189-model.pth.tar": 200_000_000,
        "auido2exp_00300-model.pth": 5_000_000,
        "auido2pose_00140-model.pth": 5_000_000,
        "BFM_model_front.mat": 100_000,
    }

import os
for d in [Config.CHECKPOINT_DIR, Config.GFPGAN_DIR, Config.BFM_DIR, 
          Config.OUTPUT_DIR, Config.INPUT_DIR]:
    os.makedirs(d, exist_ok=True)

print("⚙️  Configuration chargée :")
print(f"   Résolution SadTalker : {Config.SIZE}px")
print(f"   GFPGAN : {'✅' if Config.ENHANCE_FACE else '❌'}")
print(f"   Export : {Config.EXPORT_RESOLUTION} ({Config.BACKGROUND} background)")
print(f"   Sortie : {Config.OUTPUT_DIR}")

In [ ]:
# ─── Utilitaires et fonctions helper ──────────────────────────────────────────

def build_ffmpeg_filter(mode, width, height, background="blur"):
    """Construit un filtre ffmpeg pour redimensionner/formater la vidéo."""
    if background == "blur":
        return (
            f"split[v1][v2];"
            f"[v1]scale={width}:{height}:force_original_aspect_ratio=increase,"
            f"crop={width}:{height},boxblur=10:2[bg];"
            f"[v2]scale=iw*min({width}/iw\\,{height}/ih):ih*min({width}/iw\\,{height}/ih),"
            f"pad={width}:{height}:(ow-iw)/2:(oh-ih)/2:color=00000000[fg];"
            f"[bg][fg]overlay=0:0:format=auto,format=yuv420p"
        )
    else:
        bg_color = "black" if background == "black" else (
                   "white" if background == "white" else background.lstrip("#")
                  )
        return (
            f"scale=iw*min({width}/iw\\,{height}/ih):ih*min({width}/iw\\,{height}/ih),"
            f"pad={width}:{height}:(ow-iw)/2:(oh-ih)/2:color={bg_color},format=yuv420p"
        )

def verify_basicsr():
    """Vérifie que basicsr est correctement installé."""
    try:
        import basicsr
        from basicsr.utils.download_util import load_file_from_url
        print("✅ basicsr disponible — GFPGAN peut être activé")
        return True
    except ImportError as e:
        print(f"❌ basicsr indisponible : {e}")
        print("   → Passez ENHANCE_FACE=False pour ignorer GFPGAN")
        return False

def run_ffmpeg(input_path, output_path, filter_str, timeout=600):
    """Exécute ffmpeg avec un filtre et gestion d'erreur."""
    import subprocess
    cmd = (
        f'ffmpeg -y -i "{input_path}" -vf "{filter_str}" '
        f'-c:v libx264 -crf 18 -preset slow '
        f'-c:a aac -b:a 192k "{output_path}" -loglevel error'
    )
    try:
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
        if result.returncode != 0:
            return False, result.stderr[-500:] if result.stderr else "Unknown error"
        return True, None
    except subprocess.TimeoutExpired:
        return False, f"Timeout après {timeout}s"

# Vérification de basicsr au démarrage
print("\n🔍 Vérification de l'environnement d'export...")
has_basicsr = verify_basicsr()
if not has_basicsr and Config.ENHANCE_FACE:
    print("   ⚠️  ENHANCE_FACE=True mais basicsr manquant — force ENHANCE_FACE=False")
    Config.ENHANCE_FACE = False

---
## 🚀 Étape 6 — Génération de la vidéo

In [ ]:
# ─── Génération de la vidéo avec SadTalker ───────────────────────────────────
# SadTalker fonctionne en 3 phases internes :
#   1. Extraction des landmarks du visage source
#   2. Conversion audio → coefficients de mouvement 3D
#   3. Rendu vidéo final avec le modèle de génération de visage

import os, glob, time, subprocess
from IPython.display import HTML, display
import torch

os.chdir(Config.SADTALKER_DIR)

def log_vram(label=""):
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated(0) / 1e9
        reserved = torch.cuda.memory_reserved(0) / 1e9
        return f"{label} — VRAM : {allocated:.2f}GB utilisé, {reserved:.2f}GB réservé"
    return ""

enhancer_flag = "--enhancer gfpgan" if Config.ENHANCE_FACE else ""
bg_flag       = "--background_enhancer" if Config.BG_ENHANCER else ""

CMD = (
    f"python inference.py "
    f"--driven_audio '{DRIVEN_AUDIO}' "
    f"--source_image '{SOURCE_IMAGE}' "
    f"--result_dir '{Config.OUTPUT_DIR}' "
    f"--size {Config.SIZE} "
    f"--preprocess {Config.PREPROCESS} "
    f"--still "
    f"{enhancer_flag} "
    f"{bg_flag}"
)

print("🚀 Lancement de SadTalker...")
print(f"   Résolution : {Config.SIZE}px | Prétraitement : {Config.PREPROCESS}")
print(f"   GFPGAN     : {'activé' if Config.ENHANCE_FACE else 'désactivé'}")
print(log_vram("État initial"))
print()
print(f"⏳ Génération en cours (timeout: {Config.INFERENCE_TIMEOUT}s)...")
print("─" * 60)

t_start = time.time()
stderr_buffer = []

try:
    process = subprocess.Popen(
        CMD, shell=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True, cwd=Config.SADTALKER_DIR
    )
    
    stdout, stderr = process.communicate(timeout=Config.INFERENCE_TIMEOUT)
    
    for line in stdout.splitlines():
        print(line, flush=True)
    
    if stderr:
        stderr_buffer = stderr.splitlines()
        for line in stderr_buffer[-20:]:
            if line.strip():
                print(f"[STDERR] {line}", flush=True)
    
    returncode = process.returncode
except subprocess.TimeoutExpired:
    process.kill()
    stdout, stderr = process.communicate()
    print(f"\n❌ Timeout : dépassé {Config.INFERENCE_TIMEOUT}s.")
    print("   Conseil : réduisez Config.SIZE à '256' ou désactivez Config.ENHANCE_FACE")
    returncode = -1

elapsed = time.time() - t_start

print("─" * 60)
print(log_vram("État final"))

mp4_files = sorted(
    glob.glob(f"{Config.OUTPUT_DIR}/**/*.mp4", recursive=True),
    key=os.path.getmtime
)

if not mp4_files or returncode != 0:
    print(f"\n❌ Génération échouée (code {returncode}).")
    if stderr_buffer:
        print("\n📋 Derniers messages d'erreur :")
        for line in stderr_buffer[-10:]:
            if line.strip():
                print(f"   {line}")
    print("\n💡 Conseils : ")
    print("   • Réduisez Config.SIZE à '256' pour tester rapidement")
    print("   • Assurez-vous que l'image a un visage bien visible")
    print("   • Vérifiez que l'audio n'est pas corrompu")
else:
    RAW_VIDEO = mp4_files[-1]
    print(f"\n✅ Vidéo générée en {elapsed:.0f}s : {os.path.basename(RAW_VIDEO)}")

    display(HTML(f"""
    <video width="400" controls autoplay loop muted>
      <source src="{RAW_VIDEO}" type="video/mp4">
    </video>
    <p><em>Aperçu — format carré (avant conversion 9:16)</em></p>
    """))

---
## 📱 Étape 7 — Export vertical 9:16 pour TikTok / Reels / Shorts

In [ ]:
# ─── Conversion en format vertical 9:16 ──────────────────────────────────────
# Le format 9:16 est le standard des vidéos courtes sur :
#   TikTok, Instagram Reels, YouTube Shorts, Snapchat Spotlight

import os, time
from IPython.display import HTML, display
from google.colab import files

TARGET_W, TARGET_H = map(int, Config.EXPORT_RESOLUTION.split("x"))
EXPORT_PATH = f"{Config.OUTPUT_DIR}/sadtalker_916_{TARGET_W}x{TARGET_H}.mp4"

print(f"📱 Export en {TARGET_W}×{TARGET_H} (9:16)...")
print(f"   Fond : {Config.BACKGROUND}")
print()

filter_str = build_ffmpeg_filter("export", TARGET_W, TARGET_H, Config.BACKGROUND)
print("⏳ Conversion en cours...")
t_start = time.time()

ok, err = run_ffmpeg(RAW_VIDEO, EXPORT_PATH, filter_str, timeout=Config.FFMPEG_TIMEOUT)
elapsed = time.time() - t_start

if not ok:
    print(f"❌ Erreur ffmpeg : {err}")
    if err and "Timeout" in err:
        print("   Essayez avec une résolution plus basse (720x1280)")
else:
    size_mb = os.path.getsize(EXPORT_PATH) / 1e6
    print(f"✅ Export réussi en {elapsed:.0f}s !")
    print(f"   Fichier : {os.path.basename(EXPORT_PATH)}")
    print(f"   Taille  : {size_mb:.1f} MB")
    print(f"   Format  : {TARGET_W}×{TARGET_H} px, H.264, AAC 192kbps")

    display(HTML(f"""
    <video width="280" controls autoplay loop muted
           style="border-radius:16px; box-shadow:0 4px 20px rgba(0,0,0,0.4);">
      <source src="{EXPORT_PATH}" type="video/mp4">
    </video>
    <p><em>Aperçu — Format vertical {TARGET_W}×{TARGET_H} (9:16)</em></p>
    """))

In [ ]:
# ─── Export de variantes supplémentaires (optionnel) ─────────────────────────
# Cette cellule génère plusieurs formats en une seule passe :
#   • 9:16 Full HD  (1080×1920) — publication TikTok / Reels
#   • 9:16 HD       (720×1280)  — prévisualisation, partage rapide
#   • 1:1 carré     (1080×1080) — Instagram post, Twitter

import os

VARIANTS = [
    {"name": "TikTok_FullHD",  "w": 1080, "h": 1920, "bg": "blur"},
    {"name": "Reels_HD",       "w": 720,  "h": 1280, "bg": "blur"},
    {"name": "Carre_Instagram","w": 1080, "h": 1080, "bg": "blur"},
]

print("🎬 Génération des variantes d'export...\n")
exported = []
failed = []

for v in VARIANTS:
    dest = f"{Config.OUTPUT_DIR}/sadtalker_{v['name']}_{v['w']}x{v['h']}.mp4"
    print(f"  ⏳ {v['name']} ({v['w']}×{v['h']})...", end=" ", flush=True)
    
    filter_str = build_ffmpeg_filter("export", v["w"], v["h"], v["bg"])
    ok, err = run_ffmpeg(RAW_VIDEO, dest, filter_str, timeout=Config.FFMPEG_TIMEOUT)
    
    if ok and os.path.isfile(dest):
        size_mb = os.path.getsize(dest) / 1e6
        print(f"✅ {size_mb:.1f} MB")
        exported.append(dest)
    else:
        error_msg = err if err else "ffmpeg error"
        print(f"❌ {error_msg}")
        failed.append((v['name'], error_msg))

print(f"\n✅ {len(exported)}/{len(VARIANTS)} variantes exportées dans {Config.OUTPUT_DIR}")
if failed:
    print(f"⚠️  {len(failed)} erreur(s) :")
    for name, err in failed:
        print(f"   • {name} : {err}")

---
## 💾 Étape 8 — Téléchargement des fichiers

In [ ]:
# ─── Téléchargement des vidéos générées ──────────────────────────────────────
# Télécharge tous les fichiers MP4 présents dans le répertoire de sortie.
# Si vous ne souhaitez télécharger qu'un seul fichier, commentez les autres.

import glob, os
from google.colab import files

mp4_outputs = sorted(glob.glob(f"{OUTPUT_DIR}/*.mp4"), key=os.path.getmtime)

if not mp4_outputs:
    print("❌ Aucun fichier MP4 trouvé. Vérifiez que la génération a bien réussi.")
else:
    print(f"📥 {len(mp4_outputs)} fichier(s) prêt(s) au téléchargement :\n")
    for f_path in mp4_outputs:
        size_mb = os.path.getsize(f_path) / 1e6
        print(f"   • {os.path.basename(f_path)} — {size_mb:.1f} MB")
    print()
    print("⬇️  Téléchargement en cours...")
    for f_path in mp4_outputs:
        files.download(f_path)
    print("✅ Téléchargement lancé pour tous les fichiers.")

In [ ]:
# ─── Téléchargement sélectif (optionnel) ─────────────────────────────────────
# Utilisez cette cellule pour télécharger uniquement un fichier spécifique.
# Remplacez le nom du fichier par celui souhaité.

from google.colab import files

# Modifiez ce chemin si nécessaire :
FICHIER_A_TELECHARGER = EXPORT_PATH  # fichier 9:16 principal

if os.path.isfile(FICHIER_A_TELECHARGER):
    print(f"⬇️  Téléchargement de : {os.path.basename(FICHIER_A_TELECHARGER)}")
    files.download(FICHIER_A_TELECHARGER)
else:
    print(f"❌ Fichier introuvable : {FICHIER_A_TELECHARGER}")

---
## 🔁 Bonus — Générer une nouvelle vidéo sans tout réinstaller

In [ ]:
# ─── Régénération rapide ──────────────────────────────────────────────────────
# Si la session Colab est toujours active (environnement non réinitialisé),
# vous pouvez relancer une génération avec de nouveaux fichiers sans réinstaller.
#
# Relancez simplement les cellules dans l'ordre :
#   Étape 3 → nouvelle image
#   Étape 4 → nouveau fichier audio
#   Étape 5 → modifier les paramètres si besoin
#   Étape 6 → génération
#   Étape 7 → export 9:16
#   Étape 8 → téléchargement

print("💡 Instructions pour régénérer :")
print()
print("  1. Allez à l'Étape 3 → uploadez une nouvelle image")
print("  2. Allez à l'Étape 4 → uploadez un nouvel audio")
print("  3. Relancez les cellules Étapes 5, 6, 7, 8")
print()
print("⚠️  Si la session a été réinitialisée (runtime expired),")
print("    il faut relancer depuis l'Étape 1 (les modèles sont perdus).")
print()
print(f"📁 Fichiers actuellement dans {OUTPUT_DIR} :")
for f_path in sorted(glob.glob(f"{OUTPUT_DIR}/**", recursive=True)):
    if os.path.isfile(f_path):
        size_mb = os.path.getsize(f_path) / 1e6
        print(f"   • {os.path.relpath(f_path, OUTPUT_DIR)} — {size_mb:.1f} MB")

---
## 💡 Conseils & Astuces

### Pour un meilleur lip sync
- Utilisez une image avec le visage **bien centré** et **bien éclairé**
- Préférez une image **carrée** (512×512 ou 1024×1024)
- L'audio doit être **clair**, sans musique de fond
- La **résolution 512** donne de meilleurs résultats que 256

### Pour les réseaux sociaux
| Plateforme | Format | Résolution | Durée max |
|---|---|---|---|
| TikTok | 9:16 | 1080×1920 | 60 min |
| Instagram Reels | 9:16 | 1080×1920 | 90 s |
| YouTube Shorts | 9:16 | 1080×1920 | 60 s |
| Instagram Post | 1:1 | 1080×1080 | 60 s |
| Twitter/X | 16:9 ou 1:1 | 1280×720 | 2 min 20 s |

### Résolution des problèmes courants
- **`np.VisibleDeprecationWarning` / AttributeError numpy** → La cellule de patch (Étape 2) corrige automatiquement ce bug NumPy 2.x. Si l'erreur persiste, relancez depuis l'Étape 1.
- **CUDA out of memory** → Réduisez `SIZE` à `"256"` et désactivez `ENHANCE_FACE`
- **No face detected** → Utilisez une image avec un visage plus grand et plus net
- **Session expirée** → Relancez depuis l'Étape 1 (les modèles sont à re-télécharger)
- **Vidéo floue** → Activez `ENHANCE_FACE = True` et utilisez `SIZE = "512"`

---
*Notebook créé pour le projet David-GERBER.fr — Propulsé par [SadTalker](https://github.com/OpenTalker/SadTalker)*